In [5]:
import sys
import os
sys.path.append(os.path.abspath(".."))

import joblib
import pandas as pd

from src.data import load_matches, load_elo_baseline
from src.elo import prepare_matches, run_elo_updates, update_elo
from src.simulation import predict_match, Matchprediction
from src.features import update_team_history, update_h2h

In [16]:
draw_threshold = joblib.load('draw_threshold.joblib')
wc_model = joblib.load('world_cup_model.joblib')
history_dict = joblib.load('final_history.joblib')
h2h_dict = joblib.load('final_h2h.joblib')
country_elo = joblib.load('final_elo.joblib')

group_stage_matches = load_matches()
group_stage_matches = group_stage_matches[group_stage_matches['date'] >= "2026-06-11"]

x = predict_match('Qatar', "Switzerland", wc_model, draw_threshold, history_dict, h2h_dict, country_elo)
print(x)


Matchprediction(outcome='Draw', home_points=1, away_points=1, probability=np.float32(0.3693), diff=np.float32(0.2282))


In [22]:
#Group stage simulation

wc_groups = {
    "A": ["Canada", "Scotland", "England", "Wales"],
    "B": ["Mexico", "Jamaica", "USA", "Costa Rica"],
    "C": ["Brazil", "Argentina", "Uruguay", "Chile"],
    "D": ["Portugal", "Spain", "France", "Germany"],
    "E": ["Italy", "Netherlands", "Belgium", "Croatia"],
    "F": ["Japan", "South Korea", "Australia", "Saudi Arabia"],
    "G": ["Nigeria", "Ghana", "Senegal", "Cameroon"],
    "H": ["Egypt", "Morocco", "Algeria", "Tunisia"],
    "I": ["Sweden", "Norway", "Denmark", "Poland"],
    "J": ["Switzerland", "Austria", "Czech Republic", "Hungary"],
    "K": ["Colombia", "Paraguay", "Peru", "Ecuador"],
    "L": ["China", "Iran", "Uzbekistan", "Qatar"]
}
wc_teams = [
    "Algeria", "Argentina", "Australia", "Austria", "Belgium", 
    "Bosnia and Herzegovina", "Brazil", "Cape Verde", "Colombia", 
    "Congo DR", "Croatia", "Curaçao", "Czech Republic", "Ecuador", "Egypt", 
    "England", "France", "Germany", "Ghana", "Haiti", "Iran", "Iraq", 
    "Japan", "Jordan", "Saudi Arabia", "South Africa", "South Korea", 
    "Ivory Coast", "Mexico", "Morocco", "Netherlands", "New Zealand", 
    "Norway", "Panama", "Paraguay", "Qatar", "Scotland", "Senegal", 
    "Spain", "Sweden", "Switzerland", "Tunisia", "Türkiye", 
    "United States", "Uruguay", "Uzbekistan", "Canada"
]

rows=[]
for group, teams in wc_groups.items():
    for team in teams:
        rows.append({"Group": group,
                     "Team": team,
                     "Points": 0})
group_stage_result = pd.DataFrame(rows) #Gives us the starting group stage
i = 0
for match in group_stage_matches.itertuples(index = False):
    x = predict_match(match.home_team, match.away_team, wc_model, draw_threshold, history_dict, h2h_dict, country_elo)
    group_stage_result.loc[group_stage_result['Team'] == match.home_team, 'Points'] += x.home_points
    group_stage_result.loc[group_stage_result['Team'] == match.away_team, 'Points'] += x.away_points
    
    update_team_history(match, history_dict)
    update_h2h(match, h2h_dict)

    
    

   Group            Team  Points
0      A          Canada       0
1      A        Scotland       0
2      A         England       0
3      A           Wales       0
4      B          Mexico       0
5      B         Jamaica       0
6      B             USA       0
7      B      Costa Rica       0
8      C          Brazil       0
9      C       Argentina       0
10     C         Uruguay       0
11     C           Chile       0
12     D        Portugal       0
13     D           Spain       0
14     D          France       0
15     D         Germany       0
16     E           Italy       0
17     E     Netherlands       0
18     E         Belgium       0
19     E         Croatia       0
20     F           Japan       0
21     F     South Korea       0
22     F       Australia       0
23     F    Saudi Arabia       0
24     G         Nigeria       0
25     G           Ghana       0
26     G         Senegal       0
27     G        Cameroon       0
28     H           Egypt       0
29     H  

In [18]:

df = pd.DataFrame({
    'Name': ['Alice', 'Bob', 'Charlie'],
    'Age': [25, 30, 35]
}, index=['row1', 'row2', 'row3'])
print(df)
# Change Bob's age to 31
df.at['row2', 'Age'] = 31
print(df)


         Name  Age
row1    Alice   25
row2      Bob   30
row3  Charlie   35
         Name  Age
row1    Alice   25
row2      Bob   31
row3  Charlie   35
